### Distinct Companies — PySpark (Databricks) Notebook

Read `data/filtered_data/vulnerable_hosts.csv`, infer its schema, locate the
**company** column, and list all **distinct company names**.

This step doesn't filter anything out — every vulnerable host already has a
company name attached (from stage 2). We're just de-duplicating that list
down to one row per unique company.

### 1. Imports, config & Spark session

In [8]:
import os
import pandas as pd

from pyspark.sql import SparkSession, functions as F

try:
    spark  # provided automatically in Databricks / Jupyter-Spark
except NameError:
    spark = (
        SparkSession.builder
        .appName("distinct_companies")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "64")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

try:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
except Exception:
    PROJECT_ROOT = "/Users/vaddegurudevaraju/Desktop/CyberSecurity"

# OUTPUT_ROOT = this repo's root, one level above this notebook (pyspark_files/..).
# vulnerable_hosts.csv (read) and distinct_companies.csv (written) both live
# under OUTPUT_ROOT/data/, not the old repo.
try:
    OUTPUT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
except Exception:
    OUTPUT_ROOT = PROJECT_ROOT

HOSTS_CSV = os.path.join(OUTPUT_ROOT, "data", "filtered_data", "vulnerable_hosts.csv")

print(f"HOSTS_CSV = {HOSTS_CSV}")
print(f"exists    = {os.path.exists(HOSTS_CSV)}")

HOSTS_CSV = /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/vulnerable_hosts.csv
exists    = True


### 2. Read the CSV with `inferSchema=True`

`multiLine=True` and `escape='"'` correctly handle quoted fields that may
contain commas or newlines (common in the `summary` / `_vuln_cves` columns).

In [2]:
hosts_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .option("multiLine", True)
         .option("escape", '"')
         .csv(HOSTS_CSV)
)

print(f"rows    = {hosts_df.count():,}")
print(f"columns = {len(hosts_df.columns)}")

rows    = 205,278
columns = 23


### 3. Print the inferred schema

In [3]:
hosts_df.printSchema()

root
 |-- ip_str: string (nullable = true)
 |-- port: integer (nullable = true)
 |-- transport: string (nullable = true)
 |-- product: string (nullable = true)
 |-- version: string (nullable = true)
 |-- os: string (nullable = true)
 |-- org: string (nullable = true)
 |-- isp: string (nullable = true)
 |-- asn: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- _company: string (nullable = true)
 |-- _company_domain: string (nullable = true)
 |-- _company_source: string (nullable = true)
 |-- _company_confidence: double (nullable = true)
 |-- _company_is_provider: boolean (nullable = true)
 |-- _vuln_bucket: string (nullable = true)
 |-- _vuln_count: integer (nullable = true)
 |-- _vuln_max_cvss: double (nullable = true)
 |-- _vuln_severity: string (nullable = true)
 |-- _vuln_cves: string (nullable = true)
 |-- _vuln_verified_cves: string (nullable = true)



In [4]:
hosts_df.show(5)

+--------------+----+---------+------------+-------+------+--------------------+--------------------+--------+--------------------+-------------+----------+--------------------+--------------------+---------------+-------------------+--------------------+------------+-----------+--------------+--------------+--------------------+-------------------+
|        ip_str|port|transport|     product|version|    os|                 org|                 isp|     asn|           timestamp|      country|      city|            _company|     _company_domain|_company_source|_company_confidence|_company_is_provider|_vuln_bucket|_vuln_count|_vuln_max_cvss|_vuln_severity|          _vuln_cves|_vuln_verified_cves|
+--------------+----+---------+------------+-------+------+--------------------+--------------------+--------+--------------------+-------------+----------+--------------------+--------------------+---------------+-------------------+--------------------+------------+-----------+--------------+-

### 4. Find the company column

The CSV carries **two** company-related columns:
- `_company`        — human-readable company name (inferred by upstream attribution)
- `_company_domain` — canonical domain when available

We pick `_company` as the primary column, but list every match so it's obvious
which fields Spark discovered.

In [12]:
company_cols = [c for c in hosts_df.columns if "company" in c.lower()]
print(f"company-related columns: {company_cols}")

COMPANY_COL = "_company" if "_company" in hosts_df.columns else company_cols[0]
print(f"using COMPANY_COL = {COMPANY_COL!r}")

company-related columns: ['_company', '_company_domain', '_company_source', '_company_confidence', '_company_is_provider']
using COMPANY_COL = '_company'


In [14]:
company_cols_df = pd.DataFrame({"column_name": company_cols})
company_cols_df.head(10)

,column_name
0,_company
1,_company_domain
2,_company_source
3,_company_confidence
4,_company_is_provider


In [16]:
company_only_df = hosts_df.select(
    "_company",
    "_company_domain",
    "_company_source",
    "_company_confidence",
    "_company_is_provider",
)

print(f"rows = {company_only_df.count():,}")
company_only_df.printSchema()
company_only_df.show(10, truncate=False)

rows = 205,278
root
 |-- _company: string (nullable = true)
 |-- _company_domain: string (nullable = true)
 |-- _company_source: string (nullable = true)
 |-- _company_confidence: double (nullable = true)
 |-- _company_is_provider: boolean (nullable = true)

+------------------------------------------+---------------------+---------------+-------------------+--------------------+
|_company                                  |_company_domain      |_company_source|_company_confidence|_company_is_provider|
+------------------------------------------+---------------------+---------------+-------------------+--------------------+
|Managed Vps                               |managed-vps.net      |ssl_cert_cn_san|0.9                |false               |
|UCLOUD INFORMATION TECHNOLOGY (HK) LIMITED|NULL                 |org            |0.5                |false               |
|Linodeusercontent                         |linodeusercontent.com|hostname       |0.8                |false              

### 5. Get distinct company names

In [17]:
company_names_df = hosts_df.select(F.col("_company").alias("company_names")).distinct()

print(f"distinct companies = {company_names_df.count():,}")
company_names_df.show(50, truncate=False)

distinct companies = 48,510
+-----------------------------------------------------+
|company_names                                        |
+-----------------------------------------------------+
|Beijing Baidu Netcom Science and Technology Co., Ltd.|
|Transforips                                          |
|Synoptek, LLC                                        |
|Fubar                                                |
|The Marley Lilly LLC                                 |
|Clouding                                             |
|Bvwheel                                              |
|dus.net GmbH                                         |
|Kennergroupllc                                       |
|UAB Linama                                           |
|Gamma Telecom Limited                                |
|Dgcone                                               |
|123solden                                            |
|Beget                                                |
|Hatpaint           

### 6. Save distinct company names to CSV

Coalesce to a single partition and rename Spark's `part-*.csv` to a plain
filename so downstream tooling sees `distinct_companies.csv` directly.

In [19]:
import glob, shutil

COMPANIES_OUT = os.path.join(OUTPUT_ROOT, "data", "filtered_data", "distinct_companies.csv")

tmp_dir = COMPANIES_OUT + "__tmp"
(
    company_names_df.coalesce(1)
                    .write.mode("overwrite")
                    .option("header", True)
                    .option("escape", '"')
                    .csv(tmp_dir)
)

part = glob.glob(os.path.join(tmp_dir, "part-*.csv"))[0]
if os.path.exists(COMPANIES_OUT):
    os.remove(COMPANIES_OUT)
shutil.move(part, COMPANIES_OUT)
shutil.rmtree(tmp_dir, ignore_errors=True)

print(f"Wrote {COMPANIES_OUT}")
print(f"Size  = {os.path.getsize(COMPANIES_OUT):,} bytes")

Wrote /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/distinct_companies.csv
Size  = 655,021 bytes
